# 02. Train an adapter and evaluate it on the gold set
Runtime → Change runtime type → **T4 GPU**.

Run in order: **1, 2, 3, 4, 5, 6, 8, 9, 10, 11**. Cell 7 only after a runtime reset (run 1 and 2 first, then 7, then continue from 8).


In [1]:
# 1. Clones the project repo (or pulls the latest commit) and installs dependencies. Run it first in every new session.

REPO_URL = "https://github.com/m-mirkan/peft-cpe-extraction-pipeline.git"
BRANCH = "feat/codebook-data-curation"

import os
if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt gdown

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Cloning into '/content/repo'...
remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 170 (delta 81), reused 98 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (170/170), 2.83 MiB | 6.18 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/content/repo
Already on 'feat/codebook-data-curation'
Your branch is up to date with 'origin/feat/codebook-data-curation'.
From https://github.com/m-mirkan/peft-cpe-extraction-pipeline
 * branch            feat/codebook-data-curation -> FETCH_HEAD
Already up to date.
CUDA available: True | GPU: Tesla T4


In [2]:
# 2. Mounts Google Drive and sets the names used by every later cell (version tag, data folder, adapter folder, result files).

from google.colab import drive
drive.mount('/content/drive')

RUN = "v1"
TRAIN_DIR = "data/synthetic_v2"
ADAPTER_NAME = f"qwen25_3b_cpe_lora_{RUN}"
ADAPTER_DIR = f"checkpoints/{ADAPTER_NAME}"
OUT_P1 = f"outputs/gold_{RUN}_p1.json"
OUT_P2 = f"outputs/gold_{RUN}_p2.json"
OUT_SCORES = f"outputs/gold_{RUN}_scores.json"

DRIVE_PROJECT = "/content/drive/MyDrive/lora_projects/cpe_certificate_extraction"
DRIVE_ADAPTERS = DRIVE_PROJECT + "/adapters"
DRIVE_RESULTS = DRIVE_PROJECT + "/results"
DRIVE_ADAPTER_DIR = f"{DRIVE_ADAPTERS}/{ADAPTER_NAME}"
BATCH_SIZE, GRAD_ACCUM = 2, 8
!mkdir -p "{DRIVE_ADAPTERS}" "{DRIVE_RESULTS}"
assert ADAPTER_NAME != "qwen25_3b_cpe_lora", "refusing: that is the baseline adapter name"
print("run:", RUN, "| data:", TRAIN_DIR, "| adapter:", ADAPTER_DIR)
print("Drive folders ready:", DRIVE_ADAPTERS, "|", DRIVE_RESULTS)

Mounted at /content/drive
run: v1 | data: data/synthetic_v2 | adapter: checkpoints/qwen25_3b_cpe_lora_v1
Drive folders ready: /content/drive/MyDrive/lora_projects/cpe_certificate_extraction/adapters | /content/drive/MyDrive/lora_projects/cpe_certificate_extraction/results


In [3]:
# 3. Sanity check before training. Confirms the training files and the gold files are in the repo and shows the record counts.
import os, json
for p in [TRAIN_DIR + "/synthetic_train.jsonl", TRAIN_DIR + "/synthetic_val.jsonl",
          "data/gold/gold_inputs.json", "data/gold/gold_template.json"]:
    assert os.path.exists(p), "MISSING (did you push and pull?): " + p
assert os.path.normpath(TRAIN_DIR) != os.path.normpath("data/synthetic"), "refusing: that is the baseline training data"
n_tr = sum(1 for _ in open(TRAIN_DIR + "/synthetic_train.jsonl", encoding="utf-8"))
n_va = sum(1 for _ in open(TRAIN_DIR + "/synthetic_val.jsonl", encoding="utf-8"))
print("train records:", n_tr, "| val records:", n_va, "(v2 has 2200 / 200)")
print("first record input:", json.loads(open(TRAIN_DIR + "/synthetic_train.jsonl", encoding="utf-8").readline())["input"])
print("Training output folder:", ADAPTER_DIR)
!grep -c "save_steps" src/train_qwen.py   # must print a number >= 1 (patched trainer with resumable checkpoints)

train records: 2200 | val records: 200 (v2 has 2200 / 200)
first record input: Sybase Replication Server V15.7.1
Training output folder: checkpoints/qwen25_3b_cpe_lora_v1
3


In [4]:
# 4. Trains a LoRA adapter on the data in TRAIN_DIR. Same training settings as the baseline
!python src/train_qwen.py \
  --train_file {TRAIN_DIR}/synthetic_train.jsonl \
  --val_file {TRAIN_DIR}/synthetic_val.jsonl \
  --output_dir {DRIVE_ADAPTER_DIR} \
  --batch_size {BATCH_SIZE} --grad_accum {GRAD_ACCUM} \
  --save_steps 25 --resume

[Train] Loading tokenizer & 4-bit QLoRA model: Qwen/Qwen2.5-3B-Instruct
config.json: 100% 661/661 [00:00<00:00, 3.18MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 19.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 60.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 92.2MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 107MB/s]
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 108MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/2.20G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   1% 90.8M/6.17G [00:01<01:14, 81.5MB/s, 33.7MB/s  ]

Reconstructing (incomplete total...):  92% 5.69G/6.17G [00:36<00:03, 148MB/s,  205MB/s  ]
Reconstructing (incomplete total...):  97% 6.02G/6.17G [00:37<00:00, 174MB/s,  148MB/s  ]
Reconstructing (incomplete total...): 100% 6.17G/6.17G [00:37<00:00, 186MB/s,  174MB/s  ]

Fetching 2 files

In [5]:
# 5. After training has FINISHED, copies the final adapter files from Drive to checkpoints/ (fast local disk for inference)

import os, shutil
assert os.path.exists(DRIVE_ADAPTER_DIR + "/adapter_model.safetensors"), "final adapter not found on Drive: training (cell 4) has not finished"
os.makedirs(ADAPTER_DIR, exist_ok=True)
for f in os.listdir(DRIVE_ADAPTER_DIR):
    src = os.path.join(DRIVE_ADAPTER_DIR, f)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join(ADAPTER_DIR, f))
!cd checkpoints && zip -r -q "{DRIVE_ADAPTERS}/{ADAPTER_NAME}.zip" {ADAPTER_NAME}
print("Local adapter:", sorted(os.listdir(ADAPTER_DIR)))
print("Drive:")
!ls -la "{DRIVE_ADAPTERS}"
print("Optional: once everything works you can delete the checkpoint-* folders inside", DRIVE_ADAPTER_DIR)

Local adapter: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']
Drive:
total 110264
drwx------ 3 root root      4096 Nov 28 10:40 .
drwx------ 4 root root      4096 Nov 24 08:48 ..
drwx------ 4 root root      4096 Nov 28 10:38 qwen25_3b_cpe_lora_v1
-rw------- 1 root root 112897871 Nov 28 10:40 qwen25_3b_cpe_lora_v1.zip
Optional: once everything works you can delete the checkpoint-* folders inside /content/drive/MyDrive/lora_projects/cpe_certificate_extraction/adapters/qwen25_3b_cpe_lora_v1


In [6]:
# 6. Lists the adapter files to confirm the adapter is complete (adapter_model.safetensors + adapter_config.json).
print(sorted(os.listdir(ADAPTER_DIR)))

['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


In [7]:
# 7.  RESTORES the finished adapter from Drive. Run this ONLY in a new session / after a runtime reset
import os, shutil
if not os.path.exists(ADAPTER_DIR + "/adapter_config.json"):
    os.makedirs(ADAPTER_DIR, exist_ok=True)
    for f in os.listdir(DRIVE_ADAPTER_DIR):
        src = os.path.join(DRIVE_ADAPTER_DIR, f)
        if os.path.isfile(src):
            shutil.copy(src, os.path.join(ADAPTER_DIR, f))
print(sorted(os.listdir(ADAPTER_DIR)))

['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


In [8]:
# 8. Runs the adapter on gold titles 1-50. Predictions are saved after every title
!python src/infer_and_resolve.py --lora_dir {ADAPTER_DIR} --input_file data/gold/gold_inputs.json --offset 0 --limit 50 --output_file {OUT_P1}

[Infer] Loading base model `Qwen/Qwen2.5-3B-Instruct` on cuda (torch.bfloat16)...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:18<00:00, 23.02it/s]
[Infer] Attaching fine-tuned LoRA adapter from `checkpoints/qwen25_3b_cpe_lora_v1`...

[1] INPUT: Digital Tachograph - Vehicle Unit SE5000-8 Version E
---------------------------------------------------------------------------
  h:unknown:digital_tachograph_-_vehicle_unit_se5000-8:e [unresolved]

[2] INPUT: Xaica-Alpha PLUS ePassport Active Authentication
---------------------------------------------------------------------------
  a:xaica:alpha_plus_epassport:-                     [explicit_vendor]

[3] INPUT: DBsign for Client/Server Applications Version 3.0
---------------------------------------------------------------------------
  a:dbsign:for_client_server_applications:3.0        [explicit_vendor]

[4] INPUT: Wasion aMeter100 and aMeter300 Smart Energy Meters
---------------------

In [9]:
# 9. Runs the adapter on gold titles 51-100 -> outputs/gold_<RUN>_p2.json.
!python src/infer_and_resolve.py --lora_dir {ADAPTER_DIR} --input_file data/gold/gold_inputs.json --offset 50 --limit 50 --output_file {OUT_P2}

[Infer] Loading base model `Qwen/Qwen2.5-3B-Instruct` on cuda (torch.bfloat16)...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:11<00:00, 36.19it/s]
[Infer] Attaching fine-tuned LoRA adapter from `checkpoints/qwen25_3b_cpe_lora_v1`...

[51] INPUT: MIFARE DESFire EV3 on ST31R480 A01(version 1.0.3) (ANSSI-CC-2025/31)
---------------------------------------------------------------------------
  h:nxp:mifare_desfire_ev3:-                         [explicit_vendor]
  h:stmicroelectronics:st31r480:a01                  [explicit_vendor]
  o:nxp:firmware:1.0.3                               [clause_inheritance]

[52] INPUT: NXP JCOP 5.2 on SN100.C58 Secure Element
---------------------------------------------------------------------------
  o:nxp:jcop:5.2                                     [explicit_vendor]
  h:nxp:sn100.c58:-                                  [explicit_vendor]

[53] INPUT: EMC® Ionix™ for IT Operations Intelligence (SMARTS®) 

In [10]:
# 10. Scores the adapter on the gold set and, if the baseline files are in the repo
import json, os
n1 = len(json.load(open(OUT_P1, encoding="utf-8")))
n2 = len(json.load(open(OUT_P2, encoding="utf-8")))
print(RUN, "predictions saved:", n1, "+", n2, "(expected 50 + 50)")
cmd = f"python scripts/score_gold.py --gold data/gold/gold_template.json --pred {RUN}={OUT_P1},{OUT_P2}"
b1, b2 = "results/baseline_v0/gold_base_p1.json", "results/baseline_v0/gold_base_p2.json"
if os.path.exists(b1) and os.path.exists(b2):
    cmd += f" --pred baseline={b1},{b2}"
else:
    print("baseline files not found in results/baseline_v0/, scoring this run only")
cmd += f" --json_out {OUT_SCORES}"
!{cmd}

v1 predictions saved: 50 + 50 (expected 50 + 50)
Scored titles: 100 (status=labeled). Each title is worth 1.0 points of title-exact-match.

== ALL ==
model          titles fail      P      R     F1  partAcc exactTitle  F1-noVer
v1                100    0   0.54   0.53   0.54     0.87       0.38      0.55
baseline          100    0   0.45   0.44   0.44     0.94       0.30      0.49

== difficulty:easy ==
model          titles fail      P      R     F1  partAcc exactTitle  F1-noVer
v1                 30    0   0.55   0.60   0.57     0.95       0.60      0.60
baseline           30    0   0.57   0.57   0.57     0.94       0.57      0.60

== difficulty:hard ==
model          titles fail      P      R     F1  partAcc exactTitle  F1-noVer
v1                 29    0   0.49   0.46   0.47     0.80       0.14      0.49
baseline           29    0   0.41   0.39   0.40     0.93       0.07      0.45

== difficulty:medium ==
model          titles fail      P      R     F1  partAcc exactTitle  F1-noVer

In [11]:
# 11. Backs up this run's predictions and scores to Drive, then downloads them to your computer.
!cp {OUT_P1} {OUT_P2} {OUT_SCORES} "{DRIVE_RESULTS}/"
from google.colab import files
for f in [OUT_P1, OUT_P2, OUT_SCORES]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>